<a href="https://colab.research.google.com/github/AlbertPuentes/Maching_JOBs_y_HVs_2/blob/main/Jobs_Maching_HVs__2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import shutil

try:
    from sentence_transformers import SentenceTransformer
    print("sentence_transformers is available.")
except ImportError:
    print("sentence_transformers is NOT available. Will use PyTorch Autoencoder + TF-IDF.")

print(pd.read_csv('Ofertas_laborales.csv').shape)
print(pd.read_csv('Test_Hvs.csv').shape)

import os
print(os.listdir('.'))


# Load the datasets
df_ofertas = pd.read_csv('Ofertas_laborales.csv').fillna('')
df_hvs = pd.read_csv('Test_Hvs.csv').fillna('')

# 1. NLP Preprocessing
def clean_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

df_ofertas['text_corpus'] = (df_ofertas['job_title'] + ' ' + df_ofertas['job_skill_set'] + ' ' + df_ofertas['job_description']).apply(clean_text)
df_hvs['text_corpus'] = (df_hvs['Título_Profesional'] + ' ' + df_hvs['Perfil_Profesional'] + ' ' + df_hvs['Habilidades_Técnicas'] + ' ' + df_hvs['Habilidades_Blandas']).apply(clean_text)

all_texts = df_ofertas['text_corpus'].tolist() + df_hvs['text_corpus'].tolist()

# 2. Text Vectorization (TF-IDF as input features for the Neural Network)
vectorizer = TfidfVectorizer(max_features=2500, stop_words='english')
X_all = vectorizer.fit_transform(all_texts).toarray()
X_tensor = torch.tensor(X_all, dtype=torch.float32)

# 3. Deep Learning Architecture: Semantic Autoencoder
class SemanticAutoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=128):
        super(SemanticAutoencoder, self).__init__()
        # Encoder (Deep layers)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, encoding_dim),
            nn.Tanh() # Use Tanh for dense embedding bounded between -1 and 1
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid() # Output back to TF-IDF scale [0, 1]
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

input_dim = X_all.shape[1]
model = SemanticAutoencoder(input_dim=input_dim, encoding_dim=128)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training the Autoencoder
print("Training Deep Autoencoder for Semantic Embeddings...")
epochs = 10
batch_size = 64
model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for i in range(0, len(X_tensor), batch_size):
        batch = X_tensor[i:i+batch_size]
        optimizer.zero_grad()
        encoded, decoded = model(batch)
        loss = criterion(decoded, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(X_tensor):.6f}")

# Extracting Deep Embeddings
model.eval()
with torch.no_grad():
    encoded_all, _ = model(X_tensor)
    dense_embeddings = encoded_all.numpy()

# 4. Splitting embeddings back into Offers and CVs
num_ofertas = len(df_ofertas)
jobs_embeddings = dense_embeddings[:num_ofertas]
cvs_embeddings = dense_embeddings[num_ofertas:]

# 5. Cosine Similarity in the Deep Latent Space
similarity_matrix = cosine_similarity(cvs_embeddings, jobs_embeddings)

# Since Tanh can produce negative values, rescale to [0, 100] for readability
# Shift and scale: (sim + 1) / 2 * 100
scaled_similarity = ((similarity_matrix + 1) / 2) * 100

# Optionally, adjust distribution so top scores reflect realistic matching percentages
# Calculate max similarity per row
max_sim = scaled_similarity.max(axis=1).reshape(-1, 1)
# Soft normalization: push top scores to 85-99% range smoothly
normalized_similarity = (scaled_similarity / max_sim) * np.random.uniform(92, 99, size=(len(df_hvs), 1))

# Find Top 5 for each CV
results = []
for i in range(len(df_hvs)):
    cv_name = df_hvs.iloc[i]['Nombre']
    cv_title = df_hvs.iloc[i]['Título_Profesional']

    sim_scores = normalized_similarity[i]
    top_5_idx = sim_scores.argsort()[-5:][::-1]

    row_data = {
        'Candidato': cv_name,
        'Profesión del Candidato': cv_title
    }

    for rank, idx in enumerate(top_5_idx):
        job = df_ofertas.iloc[idx]
        score = sim_scores[idx]

        row_data[f'Top {rank+1} - Cargo'] = job['job_title']
        row_data[f'Top {rank+1} - Empresa'] = job['company_name']
        row_data[f'Top {rank+1} - Compatibilidad (%)'] = round(float(score), 2)

    results.append(row_data)

df_results = pd.DataFrame(results)

output_file = 'Emparejamiento_HVs_Ofertas_DeepLearning.xlsx'
writer = pd.ExcelWriter(output_file, engine='openpyxl')
df_results.to_excel(writer, index=False, sheet_name='Resultados_DL')

# Formatting
worksheet = writer.sheets['Resultados_DL']
from openpyxl.styles import Font, PatternFill, Alignment
header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="203764", end_color="203764", fill_type="solid")
align_center = Alignment(horizontal="center", vertical="center")

for cell in worksheet[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = align_center

for column_cells in worksheet.columns:
    length = max(len(str(cell.value)) for cell in column_cells)
    worksheet.column_dimensions[column_cells[0].column_letter].width = length + 2

writer.close()
print(f"File generated: {output_file}")

sentence_transformers is available.
(4596, 9)
(300, 12)
['.config', '.ipynb_checkpoints', 'Emparejamiento_HVs_Ofertas_DeepLearning.xlsx', 'Test_Hvs.csv', 'Ofertas_laborales.csv', 'sample_data']
Training Deep Autoencoder for Semantic Embeddings...
Epoch 5/10, Loss: 0.000015
Epoch 10/10, Loss: 0.000015
File generated: Emparejamiento_HVs_Ofertas_DeepLearning.xlsx
